# ASDEA_sensors -- MASTER tour

The single reference notebook: it shows **every method with all its inputs**
and links the 10 focused notebooks, whose examples are all represented here too.

The package is currently a skeleton (methods raise `NotImplementedError`), so
this is an API reference: the cells show the intended calls and parameters.
As each layer is implemented the cells become runnable.

Conventions: SI units (acc m/s^2, vel m/s, disp m); `ds.MOF00135.<m>` for one
sensor, `ds.<m>` to broadcast; `component="x"|"y"|"z"|"all"`; cache invalidated
automatically.

## Focused notebooks (everything here is also shown below)

| # | Notebook | Topic |
|---|----------|-------|
| 01 | `01_load_and_inspect.ipynb` | load, geometry, summary, prints |
| 02 | `02_windows_and_export.ipynb` | time windows + export to .h5 |
| 03 | `03_signal_pipeline.ipynb` | baseline / filter / derive, permutations |
| 04 | `04_filtered_vs_unfiltered.ipynb` | compare raw vs filtered spectra |
| 05 | `05_response_spectra.ipynb` | Newmark, RotD |
| 06 | `06_intensity_measures.ipynb` | Arias, CAV, Housner, peaks |
| 07 | `07_frequency_content.ipynb` | Fourier, PSD, STFT |
| 08 | `08_building_modal.ipynb` | transfer function, coherence, mode shapes, damping |
| 09 | `09_torsion_and_drift.ipynb` | torsion, orbit, drift profile, base rocking |
| 10 | `10_ambient_and_amplification.ipynb` | ambient step-by-step, HVSR, amplification |

## 0. Setup and sensor geometry

In [ ]:
import sys
sys.path.insert(0, "../src")          # run from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"   # folder with the .h5 files

In [ ]:
# Set the sensor geometry up front. Until you have surveyed UTM values you can
# put approximate ones here; this overrides config.SENSOR_GEOMETRY for the
# session. Only relative positions matter (plan distances, height differences).
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev= 0.0, azimuth=0.0)   # base -1
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev= 7.0, azimuth=0.0)   # floor 2
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5, azimuth=0.0)   # floor 3
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0, azimuth=0.0)   # floor 4 (stairwell)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0, azimuth=0.0)   # floor 4 (cantilever)
settings.SENSOR_GEOMETRY

## 1. Instantiate and inspect

In [ ]:
ds = SensorDataset(path=DATA, date_source="filename", load_mode="auto", verbose=True)
ds.verbose  = True      # internal prints, ShakerMakerResults style
ds.n_jobs   = 4
ds.parallel = True
ds.devices

In [ ]:
ds.fs; ds.dt; ds.time_span; ds.duration
ds.axes_map                 # non-standard orientation per sensor
ds.summary()
ds.cache_summary()

Internal prints

Every method prints a short line of what it did, toggled by `ds.verbose`, e.g.

```
[signal]  MOF00135  comp=all  mode=ram  files=4  n=605000  -> 39.9 MB
[filter]  MOF00135  bandpass 0.10-24.90 Hz  obspy  order=4  zerophase=True
[newmark] MOF00135  comp=x  zeta=0.05  Tmax=5.01  dT=0.01  -> 501 periods  (cached)
```

## 2. Per-sensor, broadcast, components

In [ ]:
ds.MOF00135.newmark(component="x")      # one sensor
ds.newmark(component="x")               # all sensors -> dict by device
ds.MOF00135.peaks(component="all")      # all axes -> dict by axis

## 3. Windows (start + length) and resample

In [ ]:
w = ds.MOF00135.window(start="2026-05-31 15:00:00", length="60min")   # "300sec","2hour",seconds
sub = ds.MOF00135.get_window(t0="2026-05-31 15:00", t1="2026-05-31 16:00")
ds2 = ds.resample(dt=0.01)              # whole dataset; also ds.MOF00135.resample(dt=...)

## 4. Signal pipeline -- decoupled steps

In [ ]:
sig = ds.MOF00135.signal(components="all", mode="auto", workers=4, remove_mean=False)
sig = sig.baseline(method="polynomial", components="all")
sig = sig.filter(fmin=0.1, fmax=24.9, engine="obspy", order=4, zerophase=True, components="all")
sig = sig.derive(method="trapezoid", remove_mean=True, components="all")
sig.acc_x, sig.vel_x, sig.disp_x

## 5. Composite: filtered vs unfiltered (Fourier and response spectra)

In [ ]:
raw  = ds.MOF00135.window("2026-05-31 15:00", "30min").signal().baseline()
filt = raw.filter(0.1, 24.9)

fou_raw  = raw.ambient  # placeholder; use seismic Fourier below
f_raw  = ds.MOF00135.fourier(component="x", smooth="konno")          # on raw signal
f_filt = ds.MOF00135.fourier(component="x", smooth="konno")          # on filtered signal
sp_raw  = ds.MOF00135.newmark(component="x")
sp_filt = ds.MOF00135.newmark(component="x")
# compare f_raw["spectrum"] vs f_filt["spectrum"], and sp_raw["PSa"] vs sp_filt["PSa"]

## 6. Seismic -- full inputs

In [ ]:
spec = ds.MOF00135.newmark(component="x", source="acc", zeta=0.05, max_period=5.01, dT=0.01, factor=1.0)
spec["T"]; spec["PSa"]; spec["PSv"]; spec["Sd"]; spec["Sv"]; spec["Sa"]; spec["u"]; spec["at"]
r = ds.MOF00135.rotd(comp_x="x", comp_y="y", rotd=50, damping=0.05, angle_step=5, max_period=5.01, dT=0.01)
ds.MOF00135.rotd(rotd=100)     # reuses cached matrix
ds.MOF00135.arias(component="x", low=5, high=95)
ds.MOF00135.cav(component="x"); ds.MOF00135.housner(component="x", T1=0.1, T2=2.5, zeta=0.05)
ds.MOF00135.peaks(component="all")
ds.MOF00135.fourier(component="x", num_frequencies=4, prominence=1e-6, distance_frac=0.02, smooth="konno", bexp=40)
ds.MOF00135.psd(component="x", nperseg=256, noverlap=128, window="hann", bands=[(0,1),(1,2.5),(2.5,5),(5,10)], detrend="constant")
ds.MOF00135.stft(component="x", nperseg=256, noverlap=224, window="hann", fmax=25.0)

## 7. Building -- multi-sensor

In [ ]:
ds.transfer_function(base="MOF00134", floors="all", component="x", estimator="H1", nperseg=1024, smooth="konno", fmax=25.0)
ds.coherence_matrix(component="x")
ds.modal_frequencies(component="x"); ds.mode_shapes(component="x")
ds.MOF00135.modal_tracking(component="x", window="10min", overlap=0.5, fband=(1.0,8.0), n_modes=2)
ds.torsion(floor=4, component="x")
ds.drift_profile(component="x"); ds.interstory_drift(upper="MOF00135", lower="MOF00134", component="x")
ds.base_rocking()

## 8. Ambient and amplification

In [ ]:
config = {"Fs": ds.fs, "STA":1.0, "LTA":30.0, "vent":60.0, "vmin":0.2, "vmax":2.5,
          "p":0.05, "bexp":40, "f1":0.1, "f2":25.0, "vent_seismic":False}
sig = ds.MOF00135.window("2026-05-31 15:00", "30min").signal().baseline().filter(0.1, 24.9)
amb = sig.ambient(config, component="x")
amb.sta_lta(); amb.select_windows(); amb.taper(); amb.fft(); amb.smooth(); amb.average()
amb.mean_spectrum
ds.MOF00135.hvsr(config=config, comp_h=("x","y"), comp_v="z", combine="geometric")
ds.amplification(ref="MOF00135", others=["MOF00134","MNAT0031"], basis="fourier", component="x", config=config)

## 9. Batch and sweep

In [ ]:
ds.newmark(component="x")     # parallel over devices
ds.sweep(devices="all", length="60min", overlap=0.0, analyses=["newmark","arias","ambient"], component="x", config=config)

## 10. Export a window with everything to .h5 (Provenance)

In [ ]:
win = ds.MOF00135.window("2026-05-31 15:00", "30min")
win.signal().baseline().filter(0.1, 24.9).derive()
win.newmark(component="all"); win.arias(component="all"); win.fourier(component="x")
win.export_h5("MOF00135_15-1530.h5")      # acceleration record + spectra + everything cached + Provenance
ds.export_h5("run_31MAY.h5")              # whole dataset

from asdea_sensors.io.results_file import ResultsFile
r = ResultsFile("run_31MAY.h5"); r.provenance; r.analyses("MOF00135")

## 11. Plotting

In [ ]:
ds.MOF00135.plot_signals(components="all", kind="acc")
ds.MOF00135.plot_newmark(component="x", quantity="PSa")
ds.MOF00135.plot_fourier(component="x", smooth="konno")
ds.MOF00135.plot_stft(component="x")
ds.transfer_function_plot(numerator="MOF00135", denominator="MOF00134", component="x")